# nb46 - Out-of-time tail features (H9b)

**Error analysis (2026-07-31, measured in nb45 follow-up).** Within-window time spreads: clean core 0.348 ns vs minbias 0.412 ns - the bulk of pileup is IN-TIME and unseparable at sigma_t ~ 0.25 ns. But the tails differ hugely: max|dt| median 2.1 ns (clean) vs 4.6 ns (minbias), p90 17 vs 52 ns; and timed-cell fraction 0.82 vs 0.66. Every model so far clips time features at +/-5 ns (pulls even tighter at ~1.2 ns) - a 52 ns pileup cell and a 5 ns cell look identical. The tail, which is the only separable part, is saturated away.

**Question.** Does exposing the un-saturated out-of-time tail recover the information clipping destroyed?

**Hypothesis.** H9b: adding (a) a per-cell log-scale tail feature log1p(min|dt|), (b) two per-event globals - energy fraction with |dt|>1 ns and timed-cell fraction - improves sigma_eff, most at E>17 GeV where contamination dominates.

**Research.** Belle II energy-dependent in-time gating and out-of-time fake-photon rejection (arXiv:2203.11349, /7 fake reduction); CMS HGCAL densest-window outlier rejection (EPJ Web Conf 320 00046). H9 (nb45) falsified pull REPLACEMENT; this keeps the winning raw features and ADDS tail visibility.

**Proof criterion.** Quant objective, pure minbias, 2 seeds; anchors nb43 quant 0.0445 +/- 0.0001 singles / ens 0.0437 (per-bin 0.0659/0.0479/0.0376/0.0362/0.0344/0.0387). Win = >0.002 overall or in any E>17 bin.

In [1]:
import os, sys, copy, time, pathlib
import numpy as np, pandas as pd
import torch, torch.nn as nn
REPO = pathlib.Path(os.environ['REPO_DIR']) if os.environ.get('REPO_DIR') else (
    pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd())
sys.path.insert(0, str(REPO / 'scripts'))
from run_experiments import split, resolution, PITCH, EPS
from picocal_data import build_grid, splits_for, THRESH
OUT = REPO / 'reports' / 'predictions'
DEVICE = os.environ.get('NB46_DEVICE') or ('cuda' if torch.cuda.is_available() else 'cpu')
MODE = os.environ.get('NB46_MODE', 'full')
MBF = sorted((REPO / 'data' / 'minimum_bias').glob('*.root'))
if MODE == 'smoke': MBF = MBF[:8]
t0 = time.time()
ME = build_grid(MBF, 'minbias')
print(f'device {DEVICE} | mode {MODE} | build {time.time()-t0:.0f}s')

minbias: 72554 events
device cuda | mode full | build 92s


In [2]:
W = 4; NC = 10; NG = 7
def make_windows_tail(EVS):
    rows = []; keep = []
    for i, ev in enumerate(EVS):
        m = (np.maximum(np.abs(ev['di']), np.abs(ev['dj'])) <= W) & (ev['e'] >= THRESH)
        if m.sum() < 1: continue
        di, dj, e, fr, bk, tf, tb = (v[m] for v in (ev['di'], ev['dj'], ev['e'], ev['fr'], ev['bk'], ev['tf'], ev['tb']))
        t0f = np.nanmedian(tf) if np.isfinite(tf).any() else 0.0
        t0b = np.nanmedian(tb) if np.isfinite(tb).any() else 0.0
        dtf = np.where(np.isfinite(tf), tf - t0f, np.nan)
        dtb = np.where(np.isfinite(tb), tb - t0b, np.nan)
        tfc = np.where(np.isfinite(dtf), np.clip(dtf, -5, 5), 0.0); htf = np.isfinite(dtf).astype(np.float32)
        tbc = np.where(np.isfinite(dtb), np.clip(dtb, -5, 5), 0.0); htb = np.isfinite(dtb).astype(np.float32)
        admin = np.nanmin(np.stack([np.abs(dtf), np.abs(dtb)]), 0)
        hast = np.isfinite(admin)
        logdt = np.where(hast, np.log1p(np.where(hast, admin, 0.0)), 0.0)
        oof = hast & (admin > 1.0)
        oofE = float(e[oof].sum() / (e.sum() + EPS))
        tfrac = float(hast.mean())
        rdr = np.hypot(di, dj)
        cont = np.stack([np.log1p(np.clip(e, 0, None)), np.log1p(np.clip(fr, 0, None)),
                         np.log1p(np.clip(bk, 0, None)), di.astype(np.float32), dj.astype(np.float32),
                         rdr, np.full(len(e), np.log(ev['ps'])), tfc, tbc, logdt.astype(np.float32)], 1)
        oh = np.zeros((len(e), len(PITCH)), np.float32); oh[:, ev['reg']] = 1.0
        tok = np.concatenate([cont, htf[:, None], htb[:, None], oh], 1).astype(np.float32)
        rows.append((tok, float(e.sum()), float(e.max()), ev['Etrue'], ev['reg'], oofE, tfrac))
        keep.append(i)
    return rows, np.array(keep)
rows, keep = make_windows_tail(ME)
ktr, kva, kte = splits_for(keep, len(ME))
N = len(rows); L = (2*W+1)**2; IN_DIM = rows[0][0].shape[1]
y = np.array([np.log(max(r[3], 1e-3)) for r in rows], np.float32)
Et = np.array([r[3] for r in rows], np.float32)
REG = np.array([r[4] for r in rows], int)
sumE = np.array([r[1] for r in rows], np.float32)
X = np.zeros((N, L, IN_DIM), np.float32); M = np.zeros((N, L), np.bool_)
G = np.zeros((N, NG), np.float32); Eraw = np.zeros((N, L), np.float32)
for i, (tok, se, sde, et, rg, oofE, tfrac) in enumerate(rows):
    n = tok.shape[0]; X[i, :n] = tok; M[i, :n] = True
    e = np.expm1(tok[:, 0]); Eraw[i, :n] = e
    lat = float(np.sqrt((e * tok[:, 5] ** 2).sum() / (e.sum() + EPS)))
    fbr = float(np.expm1(tok[:, 1]).sum() / (np.expm1(tok[:, 2]).sum() + EPS))
    G[i] = [np.log1p(se), np.log1p(sde), np.log(n), fbr, lat, oofE, tfrac]
la0, lb0 = np.polyfit(np.log1p(0.5 * sumE[ktr]), y[ktr], 1)
G = (G - G[ktr].mean(0)) / (G[ktr].std(0) + EPS)
cont = X[ktr][:, :, :NC].reshape(-1, NC)[M[ktr].reshape(-1)]
mean = cont.mean(0); std = cont.std(0) + EPS
X[:, :, :NC] = (X[:, :, :NC] - mean) / std; X[~M] = 0.0
T = dict(X=torch.from_numpy(X).to(DEVICE), M=torch.from_numpy(M).to(DEVICE),
         G=torch.from_numpy(G).to(DEVICE), Y=torch.from_numpy(y).unsqueeze(1).to(DEVICE),
         E=torch.from_numpy(Eraw).to(DEVICE))
oof_all = G[:, 5]
print(f'N {N}, tr/va/te {len(ktr)}/{len(kva)}/{len(kte)}, IN_DIM {IN_DIM}')

/tmp/ipykernel_494096/26564280.py:14: RuntimeWarning: All-NaN slice encountered
  admin = np.nanmin(np.stack([np.abs(dtf), np.abs(dtb)]), 0)


N 72554, tr/va/te 50787/10883/10884, IN_DIM 17


In [3]:
CFG = dict(d=128, nhead=4, layers=3, dropout=0.1, lr=3e-4, wd=1e-4, batch=96)
class SubNetT(nn.Module):
    def __init__(self, in_dim, la0, lb0):
        super().__init__()
        d = CFG['d']
        self.embed = nn.Linear(in_dim, d)
        layer = nn.TransformerEncoderLayer(d, CFG['nhead'], dim_feedforward=4*d,
                                           dropout=CFG['dropout'], batch_first=True)
        self.enc = nn.TransformerEncoder(layer, CFG['layers'], enable_nested_tensor=False)
        self.norm = nn.LayerNorm(d)
        self.head = nn.Sequential(nn.Linear(d + NG, d), nn.ReLU(), nn.Dropout(CFG['dropout']), nn.Linear(d, 3))
        self.fhead = nn.Sequential(nn.Linear(d, d // 2), nn.ReLU(), nn.Linear(d // 2, 1))
        self.la = nn.Parameter(torch.tensor(float(la0))); self.lb = nn.Parameter(torch.tensor(float(lb0)))
    def forward(self, x, m, g, ecell):
        h = self.enc(self.embed(x), src_key_padding_mask=~m)
        w = torch.sigmoid(self.fhead(h).squeeze(-1)) * m.float()
        base = self.la * torch.log1p((w * ecell).sum(1, keepdim=True)) + self.lb
        wm = m.unsqueeze(-1).float()
        p = self.norm((h * wm).sum(1) / wm.sum(1).clamp(min=1))
        return base + self.head(torch.cat([p, g], 1))
CKPT = REPO / '.scratch' / 'ckpt'; CKPT.mkdir(parents=True, exist_ok=True)
QS = torch.tensor([0.25, 0.5, 0.75], device=DEVICE)
def pinball(q, yb):
    d = yb - q
    return torch.maximum(QS * d, (QS - 1) * d).mean()
def wcalib(qv, qt, yva):
    wv = qv[:, 2] - qv[:, 0]; wt_ = qt[:, 2] - qt[:, 0]
    cuts = np.quantile(wv, [1/3, 2/3])
    gv = np.digitize(wv, cuts); gt = np.digitize(wt_, cuts)
    pe = np.empty(len(qt))
    for g in range(3):
        if (gv == g).sum() < 10 or (gt == g).sum() == 0:
            a, b2 = np.polyfit(qv[:, 1], yva, 1)
        else:
            a, b2 = np.polyfit(qv[gv == g, 1], yva[gv == g], 1)
        pe[gt == g] = np.exp(a * qt[gt == g, 1] + b2)
    return pe
def train_eval(seed, epochs, patience):
    torch.manual_seed(seed); rng = np.random.default_rng(seed)
    model = SubNetT(IN_DIM, la0, lb0).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=CFG['lr'], weight_decay=CFG['wd'])
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    ck = CKPT / f'nb46_tail_s{seed}.pt'
    def batches(idx, bs, sh):
        idx = np.asarray(idx)
        if sh: idx = rng.permutation(idx)
        for j in range(0, len(idx), bs): yield torch.from_numpy(idx[j:j+bs]).to(DEVICE)
    def fwd(b): return model(T['X'][b], T['M'][b], T['G'][b], T['E'][b])
    def run(idx):
        model.eval(); out = []
        with torch.no_grad():
            for b in batches(idx, 256, False): out.append(fwd(b).cpu().numpy())
        return np.concatenate(out)
    def vloss():
        model.eval(); s = 0.0; k = 0
        with torch.no_grad():
            for b in batches(kva, 256, False):
                s += pinball(fwd(b), T['Y'][b]).item(); k += 1
        return s / max(k, 1)
    best = 1e9; bstate = None; wait = 0; ep0 = 0
    if ck.exists():
        st = torch.load(ck, map_location=DEVICE)
        model.load_state_dict(st['model']); opt.load_state_dict(st['opt']); sched.load_state_dict(st['sched'])
        best = st['best']; bstate = st['bstate']; wait = st['wait']; ep0 = st['ep'] + 1
        rng = np.random.default_rng(seed + 1000 * ep0)
        print(f'  resume s{seed} from epoch {ep0}', flush=True)
    for ep in range(ep0, epochs):
        model.train()
        for b in batches(ktr, CFG['batch'], True):
            opt.zero_grad()
            pinball(fwd(b), T['Y'][b]).backward()
            opt.step()
        sched.step(); vv = vloss()
        if vv < best - 1e-4: best = vv; bstate = copy.deepcopy(model.state_dict()); wait = 0
        else: wait += 1
        torch.save(dict(model=model.state_dict(), opt=opt.state_dict(), sched=sched.state_dict(),
                        best=best, bstate=bstate, wait=wait, ep=ep), ck)
        if wait >= patience: break
    model.load_state_dict(bstate)
    pe = wcalib(run(kva), run(kte), y[kva])
    return float(resolution(pe, Et[kte])['sigma_eff']), pe

In [4]:
EPOCHS = {'smoke': 2, 'full': 100}[MODE]
PATIENCE = {'smoke': 99, 'full': 15}[MODE]
SEEDS = {'smoke': [0], 'full': [0, 1]}[MODE]
TAG = '' if MODE == 'full' else '_smoke'
CSVP = OUT / f'nb46_tail{TAG}.csv'
done = set()
if CSVP.exists():
    done = set(pd.read_csv(CSVP)['seed'])
    print('resume, done:', sorted(done))
for seed in SEEDS:
    if seed in done: print('skip', seed); continue
    t1 = time.time()
    sig, pe = train_eval(seed, EPOCHS, PATIENCE)
    np.save(OUT / f'nb46_pred{TAG}_tail_s{seed}.npy', pe)
    row = dict(seed=seed, sigma_eff=round(sig, 4), elapsed=round(time.time()-t1))
    pd.DataFrame([row]).to_csv(CSVP, mode='a', header=not CSVP.exists() or CSVP.stat().st_size == 0, index=False)
    print(f'tail seed {seed}: sigma_eff {sig:.4f} ({row["elapsed"]}s)', flush=True)
print(pd.read_csv(CSVP).to_string(index=False))

tail seed 0: sigma_eff 0.0442 (529s)


tail seed 1: sigma_eff 0.0461 (748s)


 seed  sigma_eff  elapsed
    0     0.0442      529
    1     0.0461      748


## Verdict vs nb43 quant anchor

Win = >0.002 overall or in any E>17 bin. Diagnostic: sigma_eff in the high-oofE tertile - if the tail features work, exactly those events should improve.

In [5]:
te_e = Et[kte]; te_r = REG[kte]
edges = np.quantile(te_e, np.linspace(0, 1, 7))
def perbin(pe):
    out = []
    for i in range(6):
        hi = edges[i+1] + (1e-9 if i == 5 else 0)
        mm = (te_e >= edges[i]) & (te_e < hi)
        out.append(resolution(pe[mm], te_e[mm])['sigma_eff'])
    return out
print('anchor nb43 quant: 0.0445 +/- 0.0001 | ens 0.0437 | per-bin 0.0659/0.0479/0.0376/0.0362/0.0344/0.0387')
SEEDS2 = {'smoke': [0], 'full': [0, 1]}[MODE]
preds = [np.load(OUT / f'nb46_pred{TAG}_tail_s{s}.npy') for s in SEEDS2
         if (OUT / f'nb46_pred{TAG}_tail_s{s}.npy').exists()]
anch = [np.load(OUT / f'nb43_pred_quant_s{s}.npy') for s in (0, 1) if (OUT / f'nb43_pred_quant_s{s}.npy').exists()]
if preds:
    sig = [resolution(p, te_e)['sigma_eff'] for p in preds]
    ens = np.stack(preds).mean(0)
    print(f'tail mean {np.mean(sig):.4f} +/- {np.std(sig):.4f} | ens {resolution(ens, te_e)["sigma_eff"]:.4f}')
    print('per-bin ' + ' / '.join(f'{b:.4f}' for b in perbin(ens)))
    oof_te = oof_all[kte]
    cuts = np.quantile(oof_te, [1/3, 2/3])
    for lo, hi, lab in [(-np.inf, cuts[0], 'low-oof'), (cuts[0], cuts[1], 'mid-oof'), (cuts[1], np.inf, 'high-oof')]:
        mm = (oof_te >= lo) & (oof_te < hi)
        line = f'{lab}: tail {resolution(ens[mm], te_e[mm])["sigma_eff"]:.4f}'
        if anch and len(anch[0]) == len(te_e):
            ea = np.stack(anch).mean(0)
            line += f' vs anchor {resolution(ea[mm], te_e[mm])["sigma_eff"]:.4f}'
        print(line)

anchor nb43 quant: 0.0445 +/- 0.0001 | ens 0.0437 | per-bin 0.0659/0.0479/0.0376/0.0362/0.0344/0.0387
tail mean 0.0452 +/- 0.0009 | ens 0.0444
per-bin 0.0679 / 0.0486 / 0.0378 / 0.0373 / 0.0363 / 0.0380
low-oof: tail 0.0433 vs anchor 0.0430
mid-oof: tail 0.0403 vs anchor 0.0405
high-oof: tail 0.0490 vs anchor 0.0468
